In [49]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, roc_curve, f1_score, precision_score, recall_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import shap
import seaborn as sns
import warnings
import math
warnings.filterwarnings('ignore')
from dagma.linear import DagmaLinear

In [50]:
from causalnex.structure.notears import from_pandas

In [51]:
from lingam import DirectLiNGAM

In [52]:
from sklearn.preprocessing import MinMaxScaler

In [53]:
from typing import Tuple

In [54]:
from castle.algorithms import GOLEM

In [55]:
df = pd.read_excel("Telco-Customer-Churn.xlsx")

In [56]:
import pickle
import copy
from causalnex.network import BayesianNetwork
from causalnex.structure import StructureModel
from causalnex.plots import plot_structure
from IPython.display import Image
from sklearn.metrics import mutual_info_score
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE

In [57]:
import json

In [58]:
import networkx as nx

In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CustomerID        7043 non-null   object 
 1   Gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   Tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [761]:
df.describe()

,SeniorCitizen,Tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [762]:
# TOTAL CHARGES TO NUMERIC
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

In [763]:
# MonthlyCharges TO NUMERIC
df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')
df['MonthlyCharges'].fillna(df['MonthlyCharges'].median(), inplace=True)

In [764]:
# Tenure TO NUMERIC
df['Tenure'] = pd.to_numeric(df['Tenure'], errors='coerce')
df['Tenure'].fillna(df['Tenure'].median(), inplace=True)

In [765]:
# DROP CUSTOMER ID
# df.drop(['CustomerID'], axis=1, inplace=True)

In [766]:
# DEFINE CATEGORICAL AND NUMERICAL
categorical_cols = df.select_dtypes(include=['object']).columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

In [767]:
print (categorical_cols)

Index(['CustomerID', 'Gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn'],
      dtype='object')


In [768]:
print (numerical_cols)

Index(['SeniorCitizen', 'Tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')


In [769]:
df[categorical_cols].nunique()  

CustomerID          7043
Gender                 2
Partner                2
Dependents             2
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
Churn                  2
dtype: int64

In [770]:
# RENAME CATEGORIES TO AVOID SPACES
for columna in categorical_cols:
    df[columna] = df[columna].astype(str).str.replace(' ', '_')

In [771]:
# ENCODE TARGET VARIABLE
df['Churn'] = df['Churn'].replace({'No': 0, 'Yes': 1})

In [772]:
# -----------------------------
# Binary categorical columns
# -----------------------------
binary_cols = [
    'Gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'PaperlessBilling'
]

# -----------------------------
# Multi-category columns
# -----------------------------
multi_cat_cols = [
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod'
]

# -----------------------------
# Numerical columns
# SeniorCitizen is binary (0,1)
# -----------------------------
numerical_cols = [
    'Tenure',
    'MonthlyCharges',
    'TotalCharges'
]

In [773]:
binary_mappings = {
    'Gender': {'Female': 0, 'Male': 1},
    'Partner': {'No': 0, 'Yes': 1},
    'Dependents': {'No': 0, 'Yes': 1},
    'PhoneService': {'No': 0, 'Yes': 1},
    'PaperlessBilling': {'No': 0, 'Yes': 1}
}

In [774]:
for col, mapping in binary_mappings.items():
    df[col] = df[col].map(mapping)


In [775]:
# ============================================
# OneHotEncoding for multi-category variables
# ============================================

ohe = OneHotEncoder(
    sparse_output=False,
    drop=None  # keep all categories for causal discovery
)

In [776]:
encoded_cat = ohe.fit_transform(df[multi_cat_cols])

In [777]:
print (encoded_cat)

[[0. 1. 0. ... 0. 1. 0.]
 [1. 0. 0. ... 0. 0. 1.]
 [1. 0. 0. ... 0. 0. 1.]
 ...
 [0. 1. 0. ... 0. 1. 0.]
 [0. 0. 1. ... 0. 0. 1.]
 [1. 0. 0. ... 0. 0. 0.]]


In [778]:
#std = StandardScaler()

In [779]:
#encoded_num = std.fit_transform(df[numerical_cols])

In [780]:
minmax = MinMaxScaler(feature_range=(0, 1))

In [781]:
encoded_num = minmax.fit_transform(df[numerical_cols])

In [782]:
print (encoded_num)

[[0.01388889 0.11542289 0.0012751 ]
 [0.47222222 0.38507463 0.21586661]
 [0.02777778 0.35422886 0.01031041]
 ...
 [0.15277778 0.11293532 0.03780868]
 [0.05555556 0.55870647 0.03321025]
 [0.91666667 0.86965174 0.78764136]]


In [783]:
# Get encoded categorical feature names
columns = ohe.get_feature_names_out(multi_cat_cols)

In [784]:
print (columns)

['MultipleLines_No' 'MultipleLines_No_phone_service' 'MultipleLines_Yes'
 'InternetService_DSL' 'InternetService_Fiber_optic' 'InternetService_No'
 'OnlineSecurity_No' 'OnlineSecurity_No_internet_service'
 'OnlineSecurity_Yes' 'OnlineBackup_No' 'OnlineBackup_No_internet_service'
 'OnlineBackup_Yes' 'DeviceProtection_No'
 'DeviceProtection_No_internet_service' 'DeviceProtection_Yes'
 'TechSupport_No' 'TechSupport_No_internet_service' 'TechSupport_Yes'
 'StreamingTV_No' 'StreamingTV_No_internet_service' 'StreamingTV_Yes'
 'StreamingMovies_No' 'StreamingMovies_No_internet_service'
 'StreamingMovies_Yes' 'Contract_Month-to-month' 'Contract_One_year'
 'Contract_Two_year' 'PaymentMethod_Bank_transfer_(automatic)'
 'PaymentMethod_Credit_card_(automatic)' 'PaymentMethod_Electronic_check'
 'PaymentMethod_Mailed_check']


In [785]:
encoded_cat_df = pd.DataFrame(
    encoded_cat,
    columns=ohe.get_feature_names_out(multi_cat_cols),
    index=df.index
)

In [786]:
encoded_num_df = pd.DataFrame(
    encoded_num,
    columns=numerical_cols,
    index=df.index
)

In [787]:
# ============================================
# Final dataset
# ============================================

df_transformed = pd.concat(
    [
        df[binary_cols],
        encoded_num_df,
        encoded_cat_df,
        df[['SeniorCitizen','CustomerID','Churn']]
    ],
    axis=1
)

In [788]:
print (df_transformed.columns)

Index(['Gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
       'Tenure', 'MonthlyCharges', 'TotalCharges', 'MultipleLines_No',
       'MultipleLines_No_phone_service', 'MultipleLines_Yes',
       'InternetService_DSL', 'InternetService_Fiber_optic',
       'InternetService_No', 'OnlineSecurity_No',
       'OnlineSecurity_No_internet_service', 'OnlineSecurity_Yes',
       'OnlineBackup_No', 'OnlineBackup_No_internet_service',
       'OnlineBackup_Yes', 'DeviceProtection_No',
       'DeviceProtection_No_internet_service', 'DeviceProtection_Yes',
       'TechSupport_No', 'TechSupport_No_internet_service', 'TechSupport_Yes',
       'StreamingTV_No', 'StreamingTV_No_internet_service', 'StreamingTV_Yes',
       'StreamingMovies_No', 'StreamingMovies_No_internet_service',
       'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One_year',
       'Contract_Two_year', 'PaymentMethod_Bank_transfer_(automatic)',
       'PaymentMethod_Credit_card_(automatic)',
     

In [790]:
df_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 42 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Gender                                   7043 non-null   int64  
 1   Partner                                  7043 non-null   int64  
 2   Dependents                               7043 non-null   int64  
 3   PhoneService                             7043 non-null   int64  
 4   PaperlessBilling                         7043 non-null   int64  
 5   Tenure                                   7043 non-null   float64
 6   MonthlyCharges                           7043 non-null   float64
 7   TotalCharges                             7043 non-null   float64
 8   MultipleLines_No                         7043 non-null   float64
 9   MultipleLines_No_phone_service           7043 non-null   float64
 10  MultipleLines_Yes                        7043 no

In [791]:
df_transformed.describe()

,Gender,Partner,Dependents,PhoneService,PaperlessBilling,Tenure,MonthlyCharges,TotalCharges,MultipleLines_No,MultipleLines_No_phone_service,...,StreamingMovies_Yes,Contract_Month-to-month,Contract_One_year,Contract_Two_year,PaymentMethod_Bank_transfer_(automatic),PaymentMethod_Credit_card_(automatic),PaymentMethod_Electronic_check,PaymentMethod_Mailed_check,SeniorCitizen,Churn
count,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,...,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.504756,0.483033,0.299588,0.903166,0.592219,0.449599,0.462803,0.261149,0.481329,0.096834,...,0.387903,0.550192,0.209144,0.240664,0.219225,0.216101,0.335794,0.228880,0.162147,0.265370
std,0.500013,0.499748,0.458110,0.295752,0.491457,0.341104,0.299403,0.261397,0.499687,0.295752,...,0.487307,0.497510,0.406726,0.427517,0.413751,0.411613,0.472301,0.420141,0.368612,0.441561
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,1.000000,0.000000,0.125000,0.171642,0.044245,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,1.000000,1.000000,0.402778,0.518408,0.159090,0.000000,0.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,0.763889,0.712438,0.434780,1.000000,0.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [792]:
print(df_transformed.head(10))

   Gender  Partner  Dependents  PhoneService  PaperlessBilling    Tenure  \
0       0        1           0             0                 1  0.013889   
1       1        0           0             1                 0  0.472222   
2       1        0           0             1                 1  0.027778   
3       1        0           0             0                 0  0.625000   
4       0        0           0             1                 1  0.027778   
5       0        0           0             1                 1  0.111111   
6       1        0           1             1                 1  0.305556   
7       0        0           0             0                 0  0.138889   
8       0        1           0             1                 1  0.388889   
9       1        0           1             1                 0  0.861111   

   MonthlyCharges  TotalCharges  MultipleLines_No  \
0        0.115423      0.001275               0.0   
1        0.385075      0.215867               1.0   
2   

In [793]:
#df_transformed.to_excel("Telco-Churn-OneHot-MinMax.xlsx", index=False)

In [ ]:
#START
df_transformed = pd.read_excel("Telco-Churn-OneHot-MinMax.xlsx")

In [61]:
# DROP CUSTOMER ID
df_transformed.drop(['CustomerID'], axis=1, inplace=True)

In [62]:
df_transformed = df_transformed.astype(float)

In [63]:
df_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 34 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Gender                                7043 non-null   float64
 1   Partner                               7043 non-null   float64
 2   Dependents                            7043 non-null   float64
 3   PhoneService                          7043 non-null   float64
 4   PaperlessBilling                      7043 non-null   float64
 5   Tenure                                7043 non-null   float64
 6   MonthlyCharges                        7043 non-null   float64
 7   TotalCharges                          7043 non-null   float64
 8   MultipleLines_No_phone_service        7043 non-null   float64
 9   MultipleLines_Yes                     7043 non-null   float64
 10  InternetService_DSL                   7043 non-null   float64
 11  InternetService_F

In [125]:
def golem_to_structure_model(
    df_transformed: pd.DataFrame,
    threshold: float = 0.05,
    equal_variances: bool = True
) -> StructureModel:
    """
    Fit GOLEM and convert the learned weighted adjacency matrix
    into a CausalNex StructureModel.
    """

    # 1. Keep numeric matrix
    X = df_transformed.astype(float).values
    columns = list(df_transformed.columns)

    # 2. Fit GOLEM
    model = GOLEM(
        equal_variances=equal_variances,
        lambda_1=0.02,
        lambda_2=5.0,
        num_iter=10000,
        learning_rate=1e-3
    )

    model.learn(X)

    # 3. Extract weighted adjacency matrix
    #W = np.asarray(model.causal_matrix)
    W = np.asarray(model.weight_causal_matrix)

    # 4. Convert to StructureModel
    sm = StructureModel()

    for col in columns:
        sm.add_node(col)

    for i, source in enumerate(columns):
        for j, target in enumerate(columns):
            weight = W[i, j]

            if i != j and abs(weight) > threshold:
                sm.add_edge(source, target, weight=float(weight))

    return sm, W



In [ ]:
#GOLEM
#sm_golem, W_golem = golem_to_structure_model(
#    df_transformed,
#    threshold=0.05
#)

In [127]:
#print("\nEdges:")
#print(list(sm_golem.edges(data=True)))

#print("\nAdjacency matrix:")
#print(W_golem.round(3))

In [128]:
#with open("model_golem.pkl", "wb") as f:
#     pickle.dump(sm_golem, f)

In [64]:
with open("model_golem.pkl", "rb") as f:
    restored_model = pickle.load(f)

In [65]:
sm = restored_model

In [66]:
num_nodes = len(sm.nodes)
print(f"Number of nodes: {num_nodes}")

Number of nodes: 34


In [67]:
weights = []
all_edges = []

for u, v, data in sm.edges(data=True):
    w = (data['weight'])
    weights.append(w)
    all_edges.append({"From": u, "To": v, "Weight": round(w, 3)})

In [68]:
stats = {
    "Statistic": ["Count", "Mean", "Median", "Std Dev", "Min", "Max"],
    "Value": [
        len(weights),
        round(pd.Series(weights).mean(), 3),
        round(pd.Series(weights).median(), 3),
        round(pd.Series(weights).std(), 3),
        round(pd.Series(weights).min(), 3),
        round(pd.Series(weights).max(), 3),
    ]
}
df_stats = pd.DataFrame(stats)

In [69]:
print(df_stats)

  Statistic    Value
0     Count  284.000
1      Mean    0.166
2    Median    0.072
3   Std Dev    1.848
4       Min   -7.597
5       Max    7.466


In [70]:
print (all_edges)

[{'From': 'Gender', 'To': 'PaperlessBilling', 'Weight': 0.089}, {'From': 'Gender', 'To': 'MonthlyCharges', 'Weight': 0.131}, {'From': 'Gender', 'To': 'TotalCharges', 'Weight': 0.192}, {'From': 'Gender', 'To': 'SeniorCitizen', 'Weight': 0.113}, {'From': 'Partner', 'To': 'Gender', 'Weight': 0.165}, {'From': 'Partner', 'To': 'Dependents', 'Weight': 0.299}, {'From': 'Partner', 'To': 'PaperlessBilling', 'Weight': 0.202}, {'From': 'Partner', 'To': 'SeniorCitizen', 'Weight': 0.124}, {'From': 'Partner', 'To': 'Churn', 'Weight': 0.163}, {'From': 'Dependents', 'To': 'Partner', 'Weight': 0.147}, {'From': 'Dependents', 'To': 'OnlineSecurity_Yes', 'Weight': 0.16}, {'From': 'Dependents', 'To': 'StreamingMovies_No_internet_service', 'Weight': 0.344}, {'From': 'PhoneService', 'To': 'Gender', 'Weight': 0.136}, {'From': 'PhoneService', 'To': 'Partner', 'Weight': 0.158}, {'From': 'PhoneService', 'To': 'PaperlessBilling', 'Weight': 0.125}, {'From': 'PhoneService', 'To': 'Tenure', 'Weight': 0.083}, {'From'

In [71]:
sm_reduced = copy.deepcopy(restored_model)

In [72]:
threshold = 0.146
edges_to_remove = [(u, v) for u, v, w in sm_reduced.edges(data=True) if abs(w["weight"]) < threshold]
sm_reduced.remove_edges_from(edges_to_remove)

In [73]:
weights = []
reduced_edges = []

for u, v, data in sm_reduced.edges(data=True):
    w = data["weight"]
    weights.append(w)
    reduced_edges.append({"From": u, "To": v, "Weight": round(w, 3)})

In [74]:
stats = {
    "Statistic": ["Count", "Mean", "Median", "Std Dev", "Min", "Max"],
    "Value": [
        len(weights),
        round(pd.Series(weights).mean(), 3),
        round(pd.Series(weights).median(), 3),
        round(pd.Series(weights).std(), 3),
        round(pd.Series(weights).min(), 3),
        round(pd.Series(weights).max(), 3),
    ]
}
df_stats = pd.DataFrame(stats)

In [75]:
print (df_stats)

  Statistic    Value
0     Count  223.000
1      Mean    0.213
2    Median    0.123
3   Std Dev    2.084
4       Min   -7.597
5       Max    7.466


In [76]:
all_edge_attributes = {
    "color":{
        "color": "#FFFFFFD9",
    },
    "font": {
            "color": 'green',
            "face" : 'arial',
            "background": '#FFFFFFD9',
            "strokeWidth": 1 ,
            "strokeColor": 'green',
            "bold": True,
            "size": 30
        },
    "length": 300,
    "label" : ""
}

In [77]:
edge_attributes = {};
for u, v, data in sm_reduced.edges(data=True):
    weight = data["weight"]
    edge_attributes[(u, v)] = {
        "color": {
            "color": "black",
            "highlight": "#79d279"
        },
        "length": 500,
        "shadow": {
            "enabled": True,
            "color": "gray"
        },
        "label": ""  # Using the edge weight as the label
    }


In [78]:
all_node_attributes = {
    "font": {
        "color": "blue",
        "face": "Helvetica",
        "size": 40
    }
}

In [79]:
plot_options = {
    "bgcolor": "white"
}

In [ ]:
viz = plot_structure(
    sm_reduced, all_edge_attributes=all_edge_attributes, edge_attributes=edge_attributes, plot_options=plot_options, all_node_attributes=all_node_attributes
)
#viz.toggle_physics(False)
viz.show("supporting_files/fully_connected_golem.html")

In [81]:
center_node = "Churn"
subgraph = nx.ego_graph(sm_reduced, center_node, radius=1, center=True, undirected = True)

In [83]:
# Remove all isolated nodes
isolated_nodes = list(nx.isolates(subgraph))
subgraph.remove_nodes_from(isolated_nodes)

In [84]:
all_edges = [(u, v, w) for u, v, w in subgraph.edges(data=True)]

In [85]:
print (all_edges)

[('Churn', 'Gender', {'origin': 'unknown', 'weight': 0.23447490355070558}), ('Churn', 'Partner', {'origin': 'unknown', 'weight': 0.1391507933586828}), ('Churn', 'PaperlessBilling', {'origin': 'unknown', 'weight': 0.35851187282881414}), ('Churn', 'SeniorCitizen', {'origin': 'unknown', 'weight': 0.2137821941393321}), ('InternetService_Fiber_optic', 'MultipleLines_Yes', {'origin': 'unknown', 'weight': -2.1219641681917953}), ('InternetService_Fiber_optic', 'OnlineBackup_Yes', {'origin': 'unknown', 'weight': -1.6599976935979122}), ('InternetService_Fiber_optic', 'Contract_Month-to-month', {'origin': 'unknown', 'weight': 0.186455600229802}), ('InternetService_Fiber_optic', 'Churn', {'origin': 'unknown', 'weight': 0.20221489904503884}), ('Gender', 'TotalCharges', {'origin': 'unknown', 'weight': 0.19177255694077375}), ('Gender', 'SeniorCitizen', {'origin': 'unknown', 'weight': 0.112665369754047}), ('MultipleLines_Yes', 'Partner', {'origin': 'unknown', 'weight': -0.18231527395906488}), ('Multip

In [86]:
edge_attributes = {};
for u, v, data in subgraph.edges(data=True):
    weight = data["weight"]
    edge_attributes[(u, v)] = {
        "color": {
            "color": "black",
            "highlight": "#79d279"
        },
        "length": 500,
        "shadow": {
            "enabled": True,
            "color": "gray"
        },
        "label": f"{weight:.3f}"  # Using the edge weight as the label
    }

In [87]:
opt = {
    "physics": {
        "enabled":False,
        "solver": "barnesHut",
        "barnesHut": {
            "gravitationalConstant": -2000,
            "centralGravity": 0.3,
            "springLength": 100,
            "springConstant": 0.04,
            "damping": 0.09,
            "avoidOverlap": 1
        }
    },
    "edges": {
        "smooth": {
            "enabled": True,
            "type": "cubicBezier",
            "forceDirection": "horizontal",
            "roundness": 0.5
        },
        "color": {
            "inherit": False
        },
        "arrows": {
            "to": {"enabled": True, "scaleFactor": 0.5}
        }
    },
    "interaction": {
        "hover": True,
        "navigationButtons": True
    },
    "layout": {
        "improvedLayout": True
    }
}

In [ ]:
viz = plot_structure(
    subgraph, all_edge_attributes=all_edge_attributes, edge_attributes=edge_attributes, plot_options=plot_options, all_node_attributes=all_node_attributes
)
viz.set_options(options=json.dumps(opt))
viz.show('supporting_files/Churn_golem.html')

In [89]:
# Remove all outbound edges from center_node
out_edges = list(subgraph.out_edges(center_node))
subgraph.remove_edges_from(out_edges)

In [ ]:
viz = plot_structure(
    subgraph, all_edge_attributes=all_edge_attributes, edge_attributes=edge_attributes, plot_options=plot_options, all_node_attributes=all_node_attributes
)
viz.set_options(options=json.dumps(opt))
viz.show('supporting_files/Churn_golem.html')

In [91]:
node = "Churn"
edges = []

In [92]:
# edges entering in Churn: X -> Churn
for source, target, data in subgraph.in_edges(node, data=True):
    edges.append({
        "Source": source,
        "Target": target,
        "Direction": "IN",
        "Weight": data.get("weight")
    })

In [93]:
df_edges = pd.DataFrame(edges)

df_edges.to_excel(
    "Global_Score_GOLEM.xlsx",
    index=False
)

In [94]:
from pathlib import Path

In [95]:
center_node = "Churn"
lambdas = [0.25, 0.50, 0.75]
output_file = Path("Path_Based_Structural_Importance_GOLEM.xlsx")

In [96]:
# ---------------------------------------------------------
# 1. Keep only direct parents of Churn
# ---------------------------------------------------------

parents = list(subgraph.predecessors(center_node))

nodes_to_keep = parents + [center_node]

In [97]:
# Induced subgraph:
# keeps Churn, its direct parents, AND all edges among parents
subgraph = subgraph.subgraph(nodes_to_keep).copy()

print(f"Number of direct parents of {center_node}: {len(parents)}")
print(f"Nodes in subgraph: {subgraph.number_of_nodes()}")
print(f"Edges in subgraph: {subgraph.number_of_edges()}")


Number of direct parents of Churn: 7
Nodes in subgraph: 8
Edges in subgraph: 22


In [113]:
viz = plot_structure(
    subgraph, all_edge_attributes=all_edge_attributes, edge_attributes=edge_attributes, plot_options=plot_options, all_node_attributes=all_node_attributes
)
viz.set_options(options=json.dumps(opt))
viz.show('supporting_files/Churn_golem_pos_only_parents.html')

supporting_files/Churn_golem_pos_only_parents.html


In [99]:
# ============================================================
# 2. HELPER FUNCTION
# ============================================================

def get_weight(G, source, target):
    return float(G[source][target].get("weight", 0.0))

In [100]:
# ============================================================
# 3. DIRECT COMPONENT D_i
# ============================================================
#
# Signed:
#     D_i = w(X_i -> Churn)
#
# Absolute:
#     |D_i|
#
# ============================================================

D_signed = {}
D_abs = {}

for node in parents:

    w = get_weight(
        subgraph,
        node,
        center_node
    )

    D_signed[node] = w
    D_abs[node] = abs(w)


In [101]:
# ============================================================
# 4. INDIRECT COMPONENT I_i
# ============================================================
#
# Only paths of EXACTLY length 2:
#
#     X_i -> X_j -> Churn
#
# Signed:
#
#     I_i_signed =
#         sum_j w_ij * w_j,Churn
#
# Absolute:
#
#     I_i_abs =
#         sum_j |w_ij * w_j,Churn|
#
# ============================================================

I_signed = {}
I_abs = {}

path_rows = []

for node_i in parents:

    signed_sum = 0.0
    abs_sum = 0.0

    for node_j in parents:

        if node_i == node_j:
            continue

        # Check for:
        #
        # node_i -> node_j -> Churn

        if (
            subgraph.has_edge(node_i, node_j)
            and
            subgraph.has_edge(node_j, center_node)
        ):

            w_ij = get_weight(
                subgraph,
                node_i,
                node_j
            )

            w_j_churn = get_weight(
                subgraph,
                node_j,
                center_node
            )

            path_signed = (
                w_ij * w_j_churn
            )

            path_abs = abs(path_signed)

            signed_sum += path_signed
            abs_sum += path_abs

            path_rows.append({
                "Source": node_i,
                "Via": node_j,
                "Target": center_node,

                "Weight_Source_Via": w_ij,
                "Weight_Via_Churn": w_j_churn,

                "Path_Signed": path_signed,
                "Path_Abs": path_abs
            })

    I_signed[node_i] = signed_sum
    I_abs[node_i] = abs_sum

In [102]:
# ============================================================
# 5. NORMALIZE D_i
# ============================================================
#
# D_abs_norm      in [0, 1]
# D_signed_norm   in [-1, 1]
#
# Both use:
#
#     max |D_i|
#
# ============================================================

max_D_abs = (
    max(D_abs.values())
    if D_abs
    else 0.0
)

D_abs_norm = {}
D_signed_norm = {}

for node in parents:

    if max_D_abs > 0:

        D_abs_norm[node] = (
            D_abs[node] / max_D_abs
        )

        D_signed_norm[node] = (
            D_signed[node] / max_D_abs
        )

    else:

        D_abs_norm[node] = 0.0
        D_signed_norm[node] = 0.0


In [103]:
# ============================================================
# 6. NORMALIZE I_i
# ============================================================
#
# Importance:
#     I_abs_norm in [0,1]
#
# Direction:
#     I_signed_norm in [-1,1]
#
# ============================================================

max_I_abs = (
    max(I_abs.values())
    if I_abs
    else 0.0
)

max_I_signed_abs = (
    max(abs(value) for value in I_signed.values())
    if I_signed
    else 0.0
)


I_abs_norm = {}
I_signed_norm = {}

for node in parents:

    if max_I_abs > 0:

        I_abs_norm[node] = (
            I_abs[node]
            / max_I_abs
        )

    else:

        I_abs_norm[node] = 0.0


    if max_I_signed_abs > 0:

        I_signed_norm[node] = (
            I_signed[node]
            / max_I_signed_abs
        )

    else:

        I_signed_norm[node] = 0.0


In [104]:
# ============================================================
# 7. CALCULATE S_i(lambda) AND C_i(lambda)
# ============================================================
#
# Structural importance:
#
# S_i(lambda) =
#     D_abs_norm
#     +
#     lambda * I_abs_norm
#
#
# Structural direction:
#
# C_i(lambda) =
#     D_signed_norm
#     +
#     lambda * I_signed_norm
#
# ============================================================

rows = []

for node in parents:

    row = {

        "Node": node,

        # Direct component
        "D_signed": D_signed[node],
        "D_abs": D_abs[node],

        "D_signed_norm": D_signed_norm[node],
        "D_abs_norm": D_abs_norm[node],

        # Indirect component
        "I_signed": I_signed[node],
        "I_abs": I_abs[node],

        "I_signed_norm": I_signed_norm[node],
        "I_abs_norm": I_abs_norm[node],
    }


    for lam in lambdas:

        # ----------------------------------------
        # IMPORTANCE
        # ----------------------------------------

        S = (
            D_abs_norm[node]
            +
            lam * I_abs_norm[node]
        )


        # ----------------------------------------
        # DIRECTION
        # ----------------------------------------

        C = (
            D_signed_norm[node]
            +
            lam * I_signed_norm[node]
        )


        row[f"S_{lam:.2f}"] = S
        row[f"C_{lam:.2f}"] = C


        # Direction of recommended intervention
        if C > 0:
            direction = "+"
        elif C < 0:
            direction = "-"
        else:
            direction = "0"

        row[f"Direction_{lam:.2f}"] = direction


    rows.append(row)


results = pd.DataFrame(rows)


In [105]:
# ============================================================
# 8. RANK S_i FOR EACH LAMBDA
# ============================================================
#
# Rank:
#     1 = highest structural importance
#
# Normalized rank:
#
#     1 = best
#     0 = worst
#
# NR = 1 - (Rank - 1)/(N - 1)
#
# ============================================================

n = len(results)

for lam in lambdas:

    S_col = f"S_{lam:.2f}"

    rank_col = f"Rank_{lam:.2f}"

    norm_rank_col = (
        f"Normalized_Rank_{lam:.2f}"
    )


    # Raw rank
    results[rank_col] = (
        results[S_col]
        .rank(
            method="min",
            ascending=False
        )
        .astype(int)
    )


    # Normalized rank
    if n > 1:

        results[norm_rank_col] = (
            1
            -
            (
                results[rank_col] - 1
            )
            /
            (n - 1)
        )

    else:

        results[norm_rank_col] = 1.0



In [106]:
# ============================================================
# 9. SORT FOR DISPLAY
# ============================================================
#
# lambda = 0.50 used ONLY for display order.
# It does not affect calculations for other lambdas.
#
# ============================================================

results = (
    results
    .sort_values(
        "Rank_0.50"
    )
    .reset_index(drop=True)
)


In [107]:
# ============================================================
# 10. TABLE WITH INDIVIDUAL LENGTH-2 PATHS
# ============================================================

paths_df = pd.DataFrame(path_rows)

if not paths_df.empty:

    paths_df = (
        paths_df
        .sort_values(
            "Path_Abs",
            ascending=False
        )
        .reset_index(drop=True)
    )


In [108]:
# ============================================================
# 11. TABLE WITH ALL EDGES OF REDUCED SUBGRAPH
# ============================================================

edge_rows = []

for source, target, data in subgraph.edges(data=True):

    edge_rows.append({

        "Source": source,
        "Target": target,

        "Weight": data.get(
            "weight",
            np.nan
        ),

        "Direct_To_Churn":
            target == center_node
    })


edges_df = pd.DataFrame(edge_rows)


In [109]:
# ============================================================
# 12. DISPLAY RESULTS
# ============================================================

display(results)

print("\nLength-2 paths:")
display(paths_df)

print("\nEdges in Churn subgraph:")
display(edges_df)

,Node,D_signed,D_abs,D_signed_norm,D_abs_norm,I_signed,I_abs,I_signed_norm,I_abs_norm,S_0.25,...,Direction_0.50,S_0.75,C_0.75,Direction_0.75,Rank_0.25,Normalized_Rank_0.25,Rank_0.50,Normalized_Rank_0.50,Rank_0.75,Normalized_Rank_0.75
0,InternetService_Fiber_optic,0.202215,0.202215,0.671178,0.671178,0.918729,0.918729,1.000000,1.000000,0.921178,...,+,1.421178,1.421178,+,2,0.833333,1,1.000000,1,1.000000
1,OnlineBackup_Yes,-0.301284,0.301284,-1.000000,1.000000,-0.011950,0.065838,-0.013007,0.071663,1.017916,...,-,1.053747,-1.009755,-,1,1.000000,2,0.833333,3,0.666667
2,TotalCharges,-0.216583,0.216583,-0.718867,0.718867,-0.225388,0.423999,-0.245326,0.461506,0.834244,...,-,1.064997,-0.902862,-,3,0.666667,3,0.666667,2,0.833333
3,Contract_Month-to-month,0.227587,0.227587,0.755393,0.755393,0.000000,0.000000,0.000000,0.000000,0.755393,...,+,0.755393,0.755393,+,4,0.500000,4,0.500000,4,0.500000
4,DeviceProtection_Yes,0.156660,0.156660,0.519974,0.519974,0.216389,0.216389,0.235531,0.235531,0.578857,...,+,0.696622,0.696622,+,6,0.166667,5,0.333333,5,0.333333
5,MultipleLines_Yes,-0.177271,0.177271,-0.588387,0.588387,-0.029654,0.029654,-0.032277,0.032277,0.596456,...,-,0.612595,-0.612595,-,5,0.333333,6,0.166667,6,0.166667
6,Partner,0.162653,0.162653,0.539865,0.539865,0.000000,0.000000,0.000000,0.000000,0.539865,...,+,0.539865,0.539865,+,7,0.000000,7,0.000000,7,0.000000



Length-2 paths:


,Source,Via,Target,Weight_Source_Via,Weight_Via_Churn,Path_Signed,Path_Abs
0,InternetService_Fiber_optic,OnlineBackup_Yes,Churn,-1.659998,-0.301284,0.500130,0.500130
1,InternetService_Fiber_optic,MultipleLines_Yes,Churn,-2.121964,-0.177271,0.376164,0.376164
2,DeviceProtection_Yes,OnlineBackup_Yes,Churn,-0.403199,-0.301284,0.121477,0.121477
3,TotalCharges,OnlineBackup_Yes,Churn,0.348799,-0.301284,-0.105087,0.105087
4,TotalCharges,InternetService_Fiber_optic,Churn,-0.465071,0.202215,-0.094044,0.094044
5,TotalCharges,Contract_Month-to-month,Churn,-0.368982,0.227587,-0.083976,0.083976
6,DeviceProtection_Yes,MultipleLines_Yes,Churn,-0.400217,-0.177271,0.070947,0.070947
7,TotalCharges,DeviceProtection_Yes,Churn,0.319724,0.156660,0.050088,0.050088
8,TotalCharges,Partner,Churn,0.302594,0.162653,0.049218,0.049218
9,InternetService_Fiber_optic,Contract_Month-to-month,Churn,0.186456,0.227587,0.042435,0.042435



Edges in Churn subgraph:


,Source,Target,Weight,Direct_To_Churn
0,InternetService_Fiber_optic,MultipleLines_Yes,-2.121964,False
1,InternetService_Fiber_optic,OnlineBackup_Yes,-1.659998,False
2,InternetService_Fiber_optic,Contract_Month-to-month,0.186456,False
3,InternetService_Fiber_optic,Churn,0.202215,True
4,MultipleLines_Yes,Partner,-0.182315,False
5,MultipleLines_Yes,Churn,-0.177271,True
6,OnlineBackup_Yes,Partner,-0.239123,False
7,OnlineBackup_Yes,MultipleLines_Yes,-0.151996,False
8,OnlineBackup_Yes,Churn,-0.301284,True
9,Contract_Month-to-month,Churn,0.227587,True


In [110]:
# ============================================================
# 13. EXPORT TO EXCEL
# ============================================================

#output_file = "Path_Based_Churn_Scores.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    results.to_excel(
        writer,
        sheet_name="Scores",
        index=False
    )

    paths_df.to_excel(
        writer,
        sheet_name="Length2_Paths",
        index=False
    )

    edges_df.to_excel(
        writer,
        sheet_name="Subgraph_Edges",
        index=False
    )


In [111]:
print(f"\nResults saved to: {output_file}")


Results saved to: Path_Based_Structural_Importance_GOLEM.xlsx
